### 1. Installation et Imports

In [2]:
import gymnasium as gym
import pygame
import tensorboard
from stable_baselines3 import DQN, PPO
import numpy as np
from stable_baselines3.common.evaluation import evaluate_policy

In [3]:
import os

# Créer les répertoires nécessaires
os.makedirs("../data/logs", exist_ok=True)
os.makedirs("../data/videos", exist_ok=True)

### 2. Exploration de l'Environnement

In [4]:
env = gym.make("LunarLander-v3")
print(f"Espace d'observation : {env.observation_space}")
print(f"Espace d'actions : {env.action_space}")

Espace d'observation : Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)
Espace d'actions : Discrete(4)


L'espace d'action est discret avec 4 actions possibles, ce qui justifie l'utilisation de l'algorithme DQN.

### 3. Définition de la Baseline (Agent Aléatoire)

In [5]:
episodes = 10
for episode in range(1, episodes+1):
    obs = env.reset()
    done = False
    score = 0
    
    while not done:
        action = env.action_space.sample()
        obs, reward, done, _, info = env.step(action)
        score += float(reward)
    print(f"Episode: {episode} Score: {score}")

Episode: 1 Score: -102.00868472106593
Episode: 2 Score: -172.20932401997868
Episode: 3 Score: -139.51613041925668
Episode: 4 Score: -360.3386956051765
Episode: 5 Score: -109.9670810074779
Episode: 6 Score: -128.37843750281533
Episode: 7 Score: -78.93578598872517
Episode: 8 Score: -135.14956207427548
Episode: 9 Score: -107.31635001338857
Episode: 10 Score: -292.3629106511242


### 4. Premier Entraînement (DQN par défaut)

In [9]:
model = DQN("MlpPolicy", env, verbose=1, tensorboard_log="../data/logs/")
model.learn(total_timesteps=100_000)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to ../data/logs/DQN_9
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 108      |
|    ep_rew_mean      | -225     |
|    exploration_rate | 0.959    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 5746     |
|    time_elapsed     | 0        |
|    total_timesteps  | 430      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 1.91     |
|    n_updates        | 82       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 101      |
|    ep_rew_mean      | -229     |
|    exploration_rate | 0.923    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 6001     |
|    time_elapsed     | 0        |
|    total_timesteps  | 810      |
|

### 4.b Entraînement avec PPO (Proximal Policy Optimization)

PPO est un algorithme de politique par gradient qui offre une meilleure stabilité durant l'entraînement. Les paramètres principaux peuvent être ajustés ci-dessous.

In [7]:
# Paramètres configurables pour PPO
ppo_params = {
    "learning_rate": 1e-3,      # Taux d'apprentissage
    "n_steps": 2048,             # Nombre d'étapes par environnement avant mise à jour
    "batch_size": 64,            # Taille du batch pour l'entraînement
    "n_epochs": 20,              # Nombre de passes sur les données collectées
    "gamma": 0.99,               # Facteur de décroissance
    "gae_lambda": 0.95,          # Facteur d'avantage exponentiel généralisé
    "clip_range": 0.2,           # Coefficient de clipping
    "ent_coef": 0.0,             # Coefficient d'entropie
}

# Créer un nouvel environnement frais pour l'entraînement PPO (sans vidéos)
env_ppo = gym.make("LunarLander-v3")

# Création du modèle PPO avec les paramètres personnalisés
model_ppo = PPO(
    "MlpPolicy", 
    env_ppo, 
    verbose=1, 
    tensorboard_log="../data/logs/",
    **ppo_params
)

# Entraînement
model_ppo.learn(total_timesteps=100_000)

print("PPO training completed!")


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to ../data/logs/PPO_3
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 96.7     |
|    ep_rew_mean     | -146     |
| time/              |          |
|    fps             | 6891     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 94.5        |
|    ep_rew_mean          | -145        |
| time/                   |             |
|    fps                  | 3843        |
|    iterations           | 2           |
|    time_elapsed         | 1           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.011796378 |
|    clip_fraction        | 0.202       |
|    clip_range           | 0.2 

### 5. Évaluation et Conclusion

In [ ]:
mean_reward_dqn, std_reward_dqn = evaluate_policy(model, env, n_eval_episodes=50)
mean_reward_ppo, std_reward_ppo = evaluate_policy(model_ppo, env_ppo, n_eval_episodes=50)

print("=" * 60)
print("RÉSULTATS DE L'ÉVALUATION")
print("=" * 60)
print(f"\nDQN:")
print(f"  Reward moyen: {mean_reward_dqn:.2f} +/- {std_reward_dqn:.2f}")

print(f"\nPPO:")
print(f"  Reward moyen: {mean_reward_ppo:.2f} +/- {std_reward_ppo:.2f}")

print(f"\nDifférence: {mean_reward_ppo - mean_reward_dqn:.2f}") # type: ignore
if mean_reward_ppo > mean_reward_dqn: # type: ignore
    print(f"✓ PPO est meilleur que DQN de {(mean_reward_ppo - mean_reward_dqn):.2f} points") # type: ignore
else:
    print(f"✓ DQN est meilleur que PPO de {(mean_reward_dqn - mean_reward_ppo):.2f} points") # type: ignore
print("=" * 60)

/Users/xaviercoulon/Documents/OC/OC_P11_Entrainement_Agent_RL/.venv/lib/python3.12/site-packages/stable_baselines3/common/evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


RÉSULTATS DE L'ÉVALUATION

DQN:
  Reward moyen: -78.41 +/- 149.34

PPO:
  Reward moyen: 195.17 +/- 68.36

Différence: 273.58
✓ PPO est meilleur que DQN de 273.58 points


 ### Exemple d'atterrissage réussi (vidéo enregistrée dans ./data/videos)

In [ ]:
import gymnasium as gym

# Création de l'env avec le mode de rendu "rgb_array" pour la vidéo
env_demo = gym.make("LunarLander-v3", render_mode="rgb_array")

# Utilisation du wrapper pour enregistrer la vidéo
env_demo = gym.wrappers.RecordVideo(
    env_demo, video_folder="../data/videos", name_prefix="heuristic-test"
)

try:
    observation, info = env_demo.reset()
    terminated = False
    truncated = False

    while not (terminated or truncated):
        # Utilisation de la fonction heuristique pour tester
        from gymnasium.envs.box2d.lunar_lander import heuristic

        action = heuristic(env_demo.unwrapped, observation)

        observation, reward, terminated, truncated, info = env_demo.step(action)
finally:
    env_demo.close()